In [ ]:
import pandas as pd
import duckdb 
from pathlib import Path

# =========================
# 1. 路径与连接
# =========================
con = duckdb.connect(database=":memory:")
output_dir = Path("output_aggregate_data")
output_dir.mkdir(parents=True, exist_ok=True)

# =========================
# 2. 读取原始数据
# =========================
user_df = con.read_parquet("output_data/01_dim_user.parquet")
date_df = con.read_parquet("output_data/02_dim_date.parquet")
task_df = con.read_parquet("output_data/03_dim_task.parquet")
event_df = con.read_parquet("output_data/03_ods_todo_event_log.parquet")

print("user rows =", user_df.count("*").fetchone()[0])
print("date rows =", date_df.count("*").fetchone()[0])
print("task rows =", task_df.count("*").fetchone()[0])
print("event rows =", event_df.count("*").fetchone()[0])

print("event types sample:")

event_type_summary = con.execute(
        "SELECT event_type, COUNT(*) AS cnt FROM read_parquet('output_data/03_ods_todo_event_log.parquet') GROUP BY 1 ORDER BY 2 DESC LIMIT 10",
        [] 
    ).fetchall()

event_type_summary

user rows = 10000
date rows = 60
task rows = 180000
event rows = 600000
event types sample:


[('create_task', 180000),
 ('app_launch', 60801),
 ('complete_task', 59673),
 ('search_task', 54412),
 ('view_list', 54399),
 ('edit_task', 51069),
 ('set_due_date', 40907),
 ('set_priority', 40842),
 ('delete_task', 20502),
 ('app_close', 20407)]

In [16]:
# =========================
# 3. 事件清洗基础视图：标准化 + 关联维度
# =========================
con.execute(
    """
    CREATE OR REPLACE VIEW v_todo_event_raw AS
    SELECT
        e.log_id AS event_id,
        e.user_id,
        e.session_id,
        CAST(e.event_time AS TIMESTAMP) AS event_time,
        CAST(e.event_time AS DATE) AS event_date,
        e.event_type,
        e.task_id,
        e.list_id,
        e.device_type,
        e.os,
        e.app_version,
        e.network_type,
        u.province,
        u.city,
        u.user_level,
        u.user_type,
        u.gender,
        u.age_group,
        u.is_active,
        u.register_channel,
        t.priority AS task_priority,
        t.due_date,
        CAST(t.create_time AS TIMESTAMP) AS task_create_time,
        CAST(t.complete_time AS TIMESTAMP) AS task_complete_time,
        t.is_completed AS task_is_completed,
        t.has_subtask,
        t.has_reminder,
        t.task_status,
        TRY_CAST(json_extract_string(e.attributes, '$.priority') AS VARCHAR) AS attr_priority,
        TRY_CAST(json_extract_string(e.attributes, '$.due_date') AS VARCHAR) AS attr_due_date,
        TRY_CAST(json_extract_string(e.attributes, '$.has_reminder') AS BOOLEAN) AS attr_has_reminder,
        TRY_CAST(json_extract_string(e.attributes, '$.has_subtask') AS BOOLEAN) AS attr_has_subtask,
        TRY_CAST(json_extract_string(e.attributes, '$.action_duration') AS INTEGER) AS action_duration,

        TRY_CAST(json_extract_string(e.attributes, '$.new_priority') AS VARCHAR) AS attr_new_priority,
        TRY_CAST(json_extract_string(e.attributes, '$.new_due_date') AS VARCHAR) AS attr_new_due_date,
        TRY_CAST(json_extract_string(e.attributes, '$.query') AS VARCHAR) AS attr_query,
        TRY_CAST(json_extract_string(e.attributes, '$.result_count') AS INTEGER) AS attr_result_count,
        TRY_CAST(json_extract_string(e.attributes, '$.feature_name') AS VARCHAR) AS attr_feature_name,
        TRY_CAST(json_extract_string(e.attributes, '$.close_reason') AS VARCHAR) AS attr_close_reason,

        e.attributes
    FROM read_parquet('output_data/03_ods_todo_event_log.parquet') AS e
    LEFT JOIN read_parquet('output_data/01_dim_user.parquet') AS u
        ON e.user_id = u.user_id
    LEFT JOIN read_parquet('output_data/03_dim_task.parquet') AS t
        ON e.task_id = t.task_id
    """,
    []
)

print("v_todo_event_raw preview:")

v_todo_event_raw = con.execute("SELECT * FROM v_todo_event_raw").fetchdf()
v_todo_event_raw


con.execute(
    "COPY v_todo_event_raw TO 'output_aggregate_data/00_v_todo_event_raw.parquet' (FORMAT PARQUET)"
)


v_todo_event_raw preview:


In [11]:
v_todo_event_raw.columns

Index(['event_id', 'user_id', 'session_id', 'event_time', 'event_date',
       'event_type', 'task_id', 'list_id', 'device_type', 'os', 'app_version',
       'network_type', 'province', 'city', 'user_level', 'user_type', 'gender',
       'age_group', 'is_active', 'register_channel', 'task_priority',
       'due_date', 'task_create_time', 'task_complete_time',
       'task_is_completed', 'has_subtask', 'has_reminder', 'task_status',
       'attr_priority', 'attr_due_date', 'attr_has_reminder',
       'attr_has_subtask', 'action_duration', 'attributes'],
      dtype='object')

In [ ]:
# 删掉冗余列，留下有用列
con.execute(
    """
    CREATE OR REPLACE TABLE v_todo_event_raw_clean AS
    SELECT
        event_id,
        user_id,
        session_id,
        event_time,
        event_date,
        event_type,
        task_id,
        list_id,
        device_type,
        os,
        app_version,
        network_type,
        province,
        city,
        user_level,
        user_type,
        gender,
        age_group,
        is_active,
        register_channel,
        task_priority,
        due_date,
        task_create_time,
        task_complete_time,
        task_is_completed,
        has_subtask,
        has_reminder,
        task_status,
        action_duration,
        attr_new_priority,
        attr_new_due_date,
        attr_query,
        attr_result_count,
        attr_feature_name,
        attr_close_reason,
        attributes
    FROM v_todo_event_raw
    """,
    []
)

print("v_todo_event_raw_clean preview:")

v_todo_event_raw_clean = con.execute("SELECT * FROM v_todo_event_raw_clean").fetchdf()

con.execute(
    "COPY v_todo_event_raw_clean TO 'output_aggregate_data/00_v_todo_event_raw_clean.parquet' (FORMAT PARQUET)"
)
v_todo_event_raw_clean

v_todo_event_raw_clean preview:


In [19]:
# =========================
# 4. DWD 1：事件明细事实表
# =========================
con.execute(
    """
    CREATE OR REPLACE TABLE dwd_todo_event_detail AS
    SELECT
        event_id,
        user_id,
        task_id,
        list_id,
        event_type,
        event_time,
        event_date,
        session_id,
        device_type,
        os,
        app_version,
        network_type,
        province,
        city,
        user_level,
        user_type,
        gender,
        age_group,
        is_active,
        register_channel,
        task_priority,
        due_date,
        task_create_time,
        task_complete_time,
        task_is_completed,
        has_subtask,
        has_reminder,
        task_status,
        action_duration,
        attr_new_priority,
        attr_new_due_date,
        attr_query,
        attr_result_count,
        attr_feature_name,
        attr_close_reason,
        attributes
    FROM v_todo_event_raw_clean
    """,
    []
)


con.execute(
    "COPY dwd_todo_event_detail TO 'output_aggregate_data/01_dwd_todo_event_detail.parquet' (FORMAT PARQUET)"
)

print("dwd_todo_event_detail rows =", con.execute("SELECT COUNT(*) FROM dwd_todo_event_detail").fetchone()[0])


dwd_todo_event_detail rows = 600000


In [20]:

# =========================
# 5. DWD 2：任务生命周期明细
# =========================
con.execute(
    """
    CREATE OR REPLACE TABLE dwd_task_lifecycle AS
    WITH task_stats AS (
        SELECT
            t.task_id,
            t.user_id,
            MIN(CASE WHEN t.event_type = 'create_task' THEN t.event_time END) AS create_time,
            MIN(CASE WHEN t.event_type = 'complete_task' THEN t.event_time END) AS first_complete_time,
            MAX(t.event_time) AS last_update_time,
            MAX(CASE WHEN t.event_type = 'complete_task' THEN 1 ELSE 0 END) AS has_complete_event,
            MAX(CASE WHEN t.event_type = 'delete_task' THEN 1 ELSE 0 END) AS has_delete_event,
            MAX(CASE WHEN t.event_type = 'update_task' THEN 1 ELSE 0 END) AS has_update_event,
            MAX(CASE WHEN t.event_type = 'remind_task' THEN 1 ELSE 0 END) AS has_remind_event
        FROM v_todo_event_raw t
        GROUP BY t.task_id, t.user_id
    )
    SELECT
        s.task_id,
        s.user_id,
        s.create_time,
        s.first_complete_time,
        s.last_update_time,
        CASE
            WHEN s.first_complete_time IS NOT NULL AND s.create_time IS NOT NULL
            THEN DATE_DIFF('second', s.create_time, s.first_complete_time)
            ELSE NULL
        END AS complete_duration_seconds,
        CASE WHEN COALESCE(s.has_complete_event, 0) = 1 THEN TRUE ELSE FALSE END AS is_completed,
        CASE
            WHEN t.due_date IS NOT NULL AND s.last_update_time IS NOT NULL
             AND CAST(s.last_update_time AS DATE) > CAST(t.due_date AS DATE)
            THEN TRUE
            ELSE FALSE
        END AS is_overdue,
        t.priority,
        t.has_reminder,
        t.has_subtask,
        t.task_status,
        t.due_date
    FROM task_stats s
    LEFT JOIN read_parquet('output_data/03_dim_task.parquet') AS t
        ON s.task_id = t.task_id
    """,
    []
)

con.execute(
    "COPY dwd_task_lifecycle TO 'output_aggregate_data/02_dwd_task_lifecycle.parquet' (FORMAT PARQUET)"
)

print("dwd_task_lifecycle rows =", con.execute("SELECT COUNT(*) FROM dwd_task_lifecycle").fetchone()[0])


dwd_task_lifecycle rows = 322790


In [21]:
# =========================
# 6. DWD 3：用户活跃明细（按天）
# =========================
con.execute(
    """
    CREATE OR REPLACE TABLE dwd_user_active_detail AS
    SELECT
        user_id,
        CAST(event_time AS DATE) AS active_date,
        MIN(event_time) AS first_event_time,
        MAX(event_time) AS last_event_time,
        COUNT(*) AS event_cnt,
        COUNT(DISTINCT session_id) AS session_cnt
    FROM v_todo_event_raw
    GROUP BY user_id, CAST(event_time AS DATE)
    ORDER BY user_id, CAST(event_time AS DATE)
    """
)

con.execute(
    "COPY dwd_user_active_detail TO 'output_aggregate_data/03_dwd_user_active_detail.parquet' (FORMAT PARQUET)"
)

print("dwd_user_active_detail rows =", con.execute("SELECT COUNT(*) FROM dwd_user_active_detail").fetchone()[0])


dwd_user_active_detail rows = 369181


In [22]:
# =========================
# 7. 预览
# =========================
print("\n--- dwd_todo_event_detail preview ---")
print(con.execute("SELECT * FROM dwd_todo_event_detail LIMIT 2").fetchdf())

print("\n--- dwd_task_lifecycle preview ---")
print(con.execute("SELECT * FROM dwd_task_lifecycle LIMIT 2").fetchdf())

print("\n--- dwd_user_active_detail preview ---")
print(con.execute("SELECT * FROM dwd_user_active_detail LIMIT 5").fetchdf())

print("\nAll DWD tables generated successfully.")


--- dwd_todo_event_detail preview ---
         event_id      user_id        task_id              list_id  \
0  log_0000103101  user_006111  task_00103101  list_user_006111_02   
1  log_0000143029  user_000861  task_00143029  list_user_000861_02   

    event_type          event_time event_date             session_id  \
0  create_task 2026-07-01 00:00:06 2026-07-01  user_006111_sess_0001   
1  create_task 2026-07-01 00:00:57 2026-07-01  user_000861_sess_0001   

  device_type       os  ... has_reminder task_status action_duration  \
0     desktop  windows  ...        False     deleted              45   
1      mobile  android  ...        False   completed              67   

  attr_new_priority attr_new_due_date attr_query attr_result_count  \
0              None              None       None              <NA>   
1              None              None       None              <NA>   

  attr_feature_name  attr_close_reason  \
0              None               None   
1              None  

In [23]:
con.close()